# 04. Annual harmonic recipe: development-only selection and freezing

Adds a third model family beside the fixed and aligned window models, which stay unchanged as the baseline. Whole-year Sentinel-2 harmonic predictors, optionally with Sentinel-1 radar, need no four-month window or green-up date. Every choice here uses development data only: spatial CV within the locked training split and holding out one development dataset at a time. Test parcels are read once, in Testing 05.

Run Preprocessing 01 first. Building the inputs samples Sentinel-1 at the selected pixels by small direct Earth Engine requests, cached under `pipeline_outputs/annual/cache/`; reruns read the cache.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from crop_pipeline.training import annual_workflow, window_variants

training_run = root / "pipeline_outputs/training/run_b58de2ed51ed"  # frozen window models (baseline)
variants_output = root / "pipeline_outputs/training/window_variants_20261122"

## Window-model follow-ups on development data

Re-scores the external study's follow-ups for each window cohort on the same development views: interval-plus-shape and shape-only predictors, parcel medians, one-setting forest changes and the matched 50/50 ensemble. A variant is adopted only if it improves the selection score consistently across cohorts.

In [ ]:
receipt = variants_output / "window_variants.json"
if not receipt.is_file():
    receipt = window_variants.evaluate(training_run, variants_output)
results = json.loads(receipt.read_text())["results"]
original = {r["cohort"]: r["selection_score"] for r in results if r["variant"] == "original"}
for r in results:
    reference = original.get(r["cohort"], max(original.get("fixed_matched", 0), original.get("aligned_matched", 0)))
    print(f"{r['cohort']:16} {r['variant']:20} selection {r['selection_score']:.3f} ({r['selection_score'] - reference:+.3f})")

## Annual inputs

All parcels from the verified quality handoff, up to 20 pixels per parcel, with development pixels shared by different parcel groups removed.

In [ ]:
inputs = annual_workflow.build_inputs(root)
record = json.loads(inputs.read_text())
print(inputs)
print({key: record[key] for key in ("pixels", "parcels", "pixels_per_parcel")})

## Select the recipe and classifier on development views

The rules are fixed before scoring:

1. **Recipe** — each recipe (Sentinel-2, Sentinel-2 + radar, and the two green-up variants) is scored with the default forest; highest mean of spatial CV and dataset-holdout macro-F1 wins, ties to fewer predictors.
2. **Settings** — on that recipe, each classifier family is tuned by spatial CV in the training split with the external study's grids (Random Forest changes, logistic regression, gradient-boosted trees); ties go to the listed default.
3. **Family** — the highest selection score wins, ties to the Random Forest.

Gradient boosting uses XGBoost when it loads (on macOS it needs `brew install libomp`); otherwise scikit-learn's histogram gradient boosting with the same grid. The backend is printed and saved.

In [ ]:
from crop_pipeline.training import classifiers

print("Gradient boosting backend:", classifiers.gradient_boosting_backend())
selection = annual_workflow.select(root)
chosen = json.loads((selection.parent / "selection.json").read_text())
print("\nRecipes (default forest):")
for name, scores in chosen["candidates"].items():
    print(f"  {name:34} predictors {scores['predictors']:3}  spatial CV {scores['spatial_cv_macro_f1']:.3f}  "
          f"dataset holdout {scores['dataset_holdout_macro_f1']:.3f}  selection {scores['selection_score']:.3f}")
print("\nClassifier families on", chosen["selected"])
for family, scores in chosen["classifier_candidates"].items():
    print(f"  {family:20} {scores['setting']:30} spatial CV {scores['spatial_cv_macro_f1']:.3f}  "
          f"dataset holdout {scores['dataset_holdout_macro_f1']:.3f}  selection {scores['selection_score']:.3f}")
print("\nSelected:", chosen["selected"], chosen["classifier"])

## Freeze

The selected recipe and classifier are fitted on the locked training split with the main pipeline's weights (a Random Forest gets 1,000 trees). Locked validation is reported for reference.

In [ ]:
models = annual_workflow.freeze(root)
info = json.loads((models.parent / "model.json").read_text())
print(models)
print("Classifier:", info["classifier"]["family"], info["classifier"]["setting"], info["classifier"]["backend"] or "")
print("Locked validation macro-F1:", round(info["validation"]["macro_f1"], 3),
      "| training majority:", info["training_majority_class"])